In [1]:
import numpy as np
import pandas as pd
from scipy import stats

In [2]:
active_validators_size = pd.read_csv('../int/active_validators_size_change.csv')
active_validators_category = pd.read_csv('../int/active_validators_category_change.csv')
active_validators_pool = pd.read_csv('../int/active_validators_pool_change.csv')

active_validators_size = active_validators_size[active_validators_size['slot'] >= 6206400]
active_validators_category = active_validators_category[active_validators_category['slot'] >= 6206400]
active_validators_pool = active_validators_pool[active_validators_pool['slot'] >= 6206400]

In [3]:
active_validators_size = active_validators_size.drop(columns=('Unnamed: 0'))
active_validators_category = active_validators_category.drop(columns=('Unnamed: 0'))
active_validators_pool = active_validators_pool.drop(columns=('Unnamed: 0'))

In [4]:
rewards_size = pd.read_csv('../int/rewards_size.csv').drop(columns=['epoch'])
rewards_category = pd.read_csv('../int/rewards_category.csv').drop(columns=['epoch'])
rewards_pool = pd.read_csv('../int/rewards_pool.csv').drop(columns=['epoch'])
rewards_size = rewards_size[rewards_size['slot'].isin(active_validators_size['slot'])]
rewards_category = rewards_category[rewards_category['slot'].isin(active_validators_category['slot'])]
rewards_pool = rewards_pool[rewards_pool['slot'].isin(active_validators_pool['slot'])]

rewards_size = rewards_size.drop(columns=('total'))
rewards_category = rewards_category.drop(columns=('total'))
rewards_pool = rewards_pool.drop(columns=('total'))

rewards_size['total'] = rewards_size.drop(columns=['slot']).mean(axis=1)
rewards_category['total'] = rewards_category.drop(columns=['slot']).mean(axis=1)
rewards_pool['total'] = rewards_pool.drop(columns=['slot']).mean(axis=1)

In [5]:
rewards_size

,1,100+,2-5,20-99,6-19,slot,total
0,3.228596,3.523359,3.328382,3.354958,3.360861,6840000.0,3.359231
1,3.191046,3.516477,3.327047,3.421507,3.397688,6847200.0,3.370753
2,3.165034,3.513273,3.310877,3.441939,3.375904,6854400.0,3.361406
3,3.160333,3.498146,3.318798,3.479943,3.481446,6861600.0,3.387733
4,3.126423,3.497919,3.337881,3.434020,3.352922,6868800.0,3.349833
...,...,...,...,...,...,...,...
294,2.672540,2.855028,2.707536,2.854167,2.794916,8956800.0,2.776838
295,2.826393,2.862574,2.734790,2.859195,2.788211,8964000.0,2.814233
296,2.690608,2.864407,2.592743,2.893811,2.807954,8971200.0,2.769905
297,2.696689,2.873085,2.662818,2.872214,2.802212,8978400.0,2.781404


In [6]:
# Calculate the percent change of APY
rewards_size_pct_change = rewards_size.set_index('slot').pct_change().reset_index()
rewards_size_pct_change

,slot,1,100+,2-5,20-99,6-19,total
0,6840000.0,NaN,NaN,NaN,NaN,NaN,NaN
1,6847200.0,-0.011630,-0.001953,-0.000401,0.019836,0.010957,0.003430
2,6854400.0,-0.008151,-0.000911,-0.004860,0.005972,-0.006411,-0.002773
3,6861600.0,-0.001485,-0.004306,0.002392,0.011041,0.031263,0.007832
4,6868800.0,-0.010730,-0.000065,0.005750,-0.013196,-0.036917,-0.011187
...,...,...,...,...,...,...,...
294,8956800.0,-0.011943,0.002562,-0.022963,-0.006187,0.017135,-0.004255
295,8964000.0,0.057568,0.002643,0.010066,0.001762,-0.002399,0.013467
296,8971200.0,-0.048042,0.000640,-0.051941,0.012107,0.007081,-0.015751
297,8978400.0,0.002260,0.003030,0.027027,-0.007463,-0.002045,0.004151


In [7]:
# Calculate the percent change of APY
rewards_size_pct_change = rewards_size.set_index('slot').pct_change().reset_index()

# Merge the two DataFrames on the slot column
price_elasticity_size = pd.merge(active_validators_size, rewards_size_pct_change, on='slot')

# Drop rows with infinite values
price_elasticity_size.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
price_elasticity_size['elasticity_total'] = price_elasticity_size['total_x'] / price_elasticity_size['total_y']
price_elasticity_size['elasticity_total'] = price_elasticity_size['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = price_elasticity_size['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column except 'total'
columns = ['1', '2-5', '6-19', '20-99', '100+']
for col in columns:
    elasticity_col_name = f'elasticity_{col}'
    price_elasticity_size[elasticity_col_name] = price_elasticity_size[f'{col}_x'] / price_elasticity_size[f'{col}_y']
    
    # Replace infinite values with NaN
    price_elasticity_size[elasticity_col_name] = price_elasticity_size[elasticity_col_name].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = price_elasticity_size[elasticity_col_name].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(price_elasticity_size)

Elasticity Analysis Results:
total: Mean Elasticity = 3.874220697439796, t(Mean) = -, SD = 42.80346980633647, N = 298, p-value = -
1: Mean Elasticity = -0.16570280744314653, t(Mean) = -1.6043018201439958, SD = 7.586318967467179, N = 298, p-value = 0.10964761361915201
2-5: Mean Elasticity = -0.4600103086372318, t(Mean) = -1.6821949492100392, SD = 12.08895392211056, N = 298, p-value = 0.09343827092560833
6-19: Mean Elasticity = -1.1733061227712074, t(Mean) = -1.6679604782771351, SD = 29.94751183536309, N = 298, p-value = 0.09591249750978324
20-99: Mean Elasticity = -21.025994096476108, t(Mean) = -1.0751778170965687, SD = 397.491098391432, N = 298, p-value = 0.28314803161013674
100+: Mean Elasticity = -27.36047789795979, t(Mean) = -1.092494967205122, SD = 491.68454187807527, N = 298, p-value = 0.2754877746947302
          slot       1_x    100+_x     2-5_x   20-99_x    6-19_x   total_x  \
0    6840000.0  0.030026  0.012751  0.000000  0.033270  0.000000  0.013430   
1    6847200.0  0.00000

In [8]:
# Calculate the percent change of APY
rewards_category_pct_change = rewards_category.set_index('slot').pct_change().reset_index()

# Merge the two DataFrames on the slot column
price_elasticity_category = pd.merge(active_validators_category, rewards_category_pct_change, on='slot')

# Drop rows with infinite values
price_elasticity_category.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
price_elasticity_category['elasticity_total'] = price_elasticity_category['total_x'] / price_elasticity_category['total_y']
price_elasticity_category['elasticity_total'] = price_elasticity_category['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = price_elasticity_category['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column
columns = [col for col in active_validators_category.columns if col != 'slot']
for col in columns:
    elasticity_col_name = f'elasticity_{col}'
    price_elasticity_category[elasticity_col_name] = price_elasticity_category[f'{col}_x'] / price_elasticity_category[f'{col}_y']
    
    # Replace infinite values with NaN
    price_elasticity_category[elasticity_col_name] = price_elasticity_category[elasticity_col_name].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = price_elasticity_category[elasticity_col_name].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(price_elasticity_category)

Elasticity Analysis Results:
total: Mean Elasticity = 0.5039328988207037, t(Mean) = 0.0, SD = 25.57165100494027, N = 298, p-value = 1.0
CEX: Mean Elasticity = 2.5640786216133153, t(Mean) = 1.0775062040615686, SD = 20.86752455903095, N = 298, p-value = 0.28170915196667123
Liquid Restaking: Mean Elasticity = -27.947651506453397, t(Mean) = -1.02413063466853, SD = 478.89573635962984, N = 298, p-value = 0.30660227262950646
Liquid Staking: Mean Elasticity = -0.8883209326293365, t(Mean) = -0.6675732008919365, SD = 25.342470874526942, N = 298, p-value = 0.5046653588962836
Solo Stakers: Mean Elasticity = 1.2194545440358595, t(Mean) = 0.36943836141713066, SD = 21.538941714945683, N = 298, p-value = 0.7119364728228926
Staking Pools: Mean Elasticity = 0.12641864312107642, t(Mean) = -0.201354823424375, SD = 19.839415571778915, N = 298, p-value = 0.840494324589202
Unidentified: Mean Elasticity = -4.127574023237339, t(Mean) = -0.5813800735131353, SD = 135.1230108005103, N = 298, p-value = 0.561395860

In [9]:
# Calculate the percent change of APY
rewards_pool_pct_change = rewards_pool.set_index('slot').pct_change().reset_index()

# Merge the two DataFrames on the slot column
apy_elasticity_pool = pd.merge(active_validators_pool, rewards_pool_pct_change, on='slot')

# Drop rows with infinite values
apy_elasticity_pool.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
apy_elasticity_pool['elasticity_total'] = apy_elasticity_pool['total_x'] / apy_elasticity_pool['total_y']
apy_elasticity_pool['elasticity_total'] = apy_elasticity_pool['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = apy_elasticity_pool['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column
columns = ['Lido', 'Coinbase', 'Binance', 'Rocketpool', 'Kraken', 'OKX', 'Bitcoin Suisse', 'Ledger Live', 'Ether.Fi', 'Mantle', 'Other Stakers']
for col in columns:
    elasticity_col_name = f'elasticity_{col}'
    apy_elasticity_pool[elasticity_col_name] = apy_elasticity_pool[f'{col}_x'] / apy_elasticity_pool[f'{col}_y']
    
    # Replace infinite values with NaN
    apy_elasticity_pool[elasticity_col_name] = apy_elasticity_pool[elasticity_col_name].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = apy_elasticity_pool[elasticity_col_name].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(apy_elasticity_pool)

Elasticity Analysis Results:
total: Mean Elasticity = -0.5662594965840174, t(Mean) = -, SD = 34.02161362133796, N = 298, p-value = -
Lido: Mean Elasticity = 0.10090136041816589, t(Mean) = 0.3187507086273136, SD = 12.166525925363704, N = 298, p-value = 0.7500945753347888
Coinbase: Mean Elasticity = 2.35908168609699, t(Mean) = 1.0568196847799747, SD = 33.553743852240956, N = 298, p-value = 0.2910235809718344
Binance: Mean Elasticity = 1.615037472876148, t(Mean) = 0.9594985416396394, SD = 19.561677602007418, N = 298, p-value = 0.3377967898262165
Rocketpool: Mean Elasticity = 1.3612439454581864, t(Mean) = 0.9465327230611159, SD = 8.848343987542965, N = 298, p-value = 0.344555134310196
Kraken: Mean Elasticity = 0.9299428273994932, t(Mean) = 0.6262207948241151, SD = 23.3168865708738, N = 298, p-value = 0.5314419970817419
OKX: Mean Elasticity = 0.14881756834375917, t(Mean) = 0.3573328045555709, SD = 5.991890073354308, N = 298, p-value = 0.7210816490595608
Bitcoin Suisse: Mean Elasticity = -0.

/var/folders/mb/5hm6pgrs3zj_1m_kgvpt40jw0000gn/T/ipykernel_24027/3842336488.py:2: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  rewards_pool_pct_change = rewards_pool.set_index('slot').pct_change().reset_index()
